In [47]:
%pip install groq
%pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [48]:
# Biblioteca para manipular variáveis de ambiente, e manipulação de arquivos e diretórios

import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv() # Carrega as variáveis do arquivo .env para o os.environ

True

In [49]:
# Criação do cliente Groq utilizando a chave da API armazenada na variável de ambiente
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))


# Criação de uma solicitação de conclusão de chat utilizando o modelo "llama-3.3-70b-versatile" e uma mensagem do usuário.
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explique a importância de modelos de linguagem rápidos",
        }
    ],
    # Especificação do modelo a ser utilizado para a geração da resposta.
    model="llama-3.3-70b-versatile",
)

# Impressão do conteúdo da resposta gerada pelo modelo.
print(chat_completion.choices[0].message.content)

**Introdução**

Os modelos de linguagem rápidos são um tipo de modelo de linguagem que se destaca por sua capacidade de processar e gerar texto de forma eficiente e rápida. Esses modelos são treinados em grandes conjuntos de dados e utilizam algoritmos avançados para aprender a estrutura e o significado da linguagem. Neste artigo, vamos explorar a importância dos modelos de linguagem rápidos e como eles estão revolucionando a forma como interagimos com a tecnologia.

**Vantagens dos Modelos de Linguagem Rápidos**

1. **Eficiência**: Os modelos de linguagem rápidos são capazes de processar e gerar texto em tempo real, o que é fundamental para aplicações que exigem uma resposta rápida, como chats de suporte ao cliente, interfaces de voz e tradução de texto em tempo real.
2. **Escalabilidade**: Esses modelos podem ser facilmente escalados para lidar com grandes volumes de dados e requisições, tornando-os ideais para aplicações de larga escala, como processamento de texto em massa e anális

In [50]:
# Definição de uma classe Agent para implementação de um agente que pode interagir com o modelo de linguagem para realizar tarefas específicas.
class Agent:
    # funcão de inicialização da classe Agent, que recebe um cliente Groq e uma mensagem de sistema opcional para configurar o comportamento do agente.
    def __init__(self, client, system):
        self.client = client
        self.system = system
        self.messages = []
        if self.system is not None:
            self.messages.append({"role": "system", "content": self.system})
    # Método para processar uma mensagem do usuário, adicionar a mensagem à lista de mensagens, executar a solicitação de chat e retornar a resposta gerada pelo modelo.
    def __call__(self, message=""):
        if message:
            self.messages.append({"role": "user", "content": message})
        resultado = self.executar()
        self.messages.append({"role": "assistant", "content": resultado})
        return resultado
    
    # Método para executar a solicitação de chat utilizando o cliente Groq e retornar a resposta gerada pelo modelo.
    def executar(self):
        completion = client.chat.completions.create(
            messages= self.messages,
            model="llama-3.3-70b-versatile",
        )
        return completion.choices[0].message.content


In [51]:
# Definição de um prompt de sistema para o agente, que orienta o comportamento do agente em um loop de pensamento, ação, pausa e observação, e define as ações disponíveis para o agente executar com base nas perguntas recebidas.       
system_prompt = """
Você executa em um loop de Pensamento, Ação, PAUSA, Observação.
Ao final do loop você produz uma Resposta
Use Pensamento para descrever seus pensamentos sobre a pergunta que você recebeu.
Use Ação para executar uma das ações disponíveis para você - depois retorne PAUSA.
Observação será o resultado da execução dessas ações.

Suas ações disponíveis são:

calcular_funcao_2ograu:
ex. calcular_funcao_2ograu: a , b, c
Executa a fórmula de Bhaskara para resolver uma equação quadrática com os coeficientes a, b e c fornecidos e retorna o resultado

calcular_tempo_luz:
ex. calcular_tempo_luz: Mars, Earth
Calcula o tempo que a luz leva para viajar entre a origem (padrão: Earth) e o planeta informado, retornando tempo em segundos e minutos e a distância aproximada.

Sessão de exemplo:

Pergunta: Quanto tempo a luz leva para viajar de Marte até a Terra?
Pensamento: Preciso calcular o tempo que a luz leva para viajar entre Marte e a Terra usando a ação disponível
Ação: calcular_tempo_luz: Mars, Earth
PAUSA

Você será chamado novamente com isso:

Observação: tempo = 187.4, distancia = 5.62e10

Se tiver a resposta, produza-a como a Resposta.

Resposta: A luz leva aproximadamente 187.4 segundos (ou cerca de 3.12 minutos) para viajar de Marte até a Terra, cobrindo uma distância aproximada de 5.62 × 10^10 km.

Agora é sua vez:
""".strip()

In [52]:
import math

# Função para calcular as raízes de uma equação do 2º grau (ax² + bx + c = 0).
# Aceita:
# - três argumentos numéricos (a, b, c), ou
# - uma string no formato "a, b, c".
def calcular_funcao_2ograu(a, b=None, c=None):
    # Suporta chamada com string única no formato "a, b, c"
    if isinstance(a, str) and b is None and c is None:
        partes = [p.strip() for p in a.split(",")]

        # Valida formato esperado
        if len(partes) != 3:
            return {"erro": "Use o formato: a, b, c"}

        # Converte valores para float
        a, b, c = map(float, partes)
    else:
        # Suporta chamada tradicional com 3 argumentos
        a, b, c = float(a), float(b), float(c)

    # Em equação de 2º grau, 'a' não pode ser zero
    if a == 0:
        return {"erro": "Coeficiente 'a' deve ser diferente de 0"}

    # Calcula discriminante (delta)
    delta = b**2 - 4 * a * c

    # Retorna raízes conforme o valor de delta
    if delta > 0:
        # Duas raízes reais distintas
        x1 = (-b + math.sqrt(delta)) / (2 * a)
        x2 = (-b - math.sqrt(delta)) / (2 * a)
        return {"delta": delta, "x1": x1, "x2": x2}
    elif delta == 0:
        # Uma raiz real (raiz dupla)
        x = -b / (2 * a)
        return {"delta": delta, "x": x}
    else:
        # Duas raízes complexas conjugadas
        real = -b / (2 * a)
        imag = math.sqrt(-delta) / (2 * a)
        return {"delta": delta, "x1": complex(real, imag), "x2": complex(real, -imag)}


# Função para calcular o tempo que a luz leva entre dois planetas.
# Aceita:
# - dois argumentos: destino, origem (origem padrão = "Earth"), ou
# - uma string única: "destino, origem".
def calcular_tempo_luz(destino, origem="Earth"):
    # Permite chamada com string única: "destino, origem"
    if isinstance(destino, str) and "," in destino and origem == "Earth":
        partes = [p.strip() for p in destino.split(",")]
        if len(partes) == 2:
            destino, origem = partes[0], partes[1]

    # Distância média ao Sol em quilômetros (aproximações)
    distancia_sol_km = {
        "mercury": 57_900_000,
        "venus": 108_200_000,
        "earth": 149_600_000,
        "mars": 227_900_000,
        "jupiter": 778_500_000,
        "saturn": 1_433_000_000,
        "uranus": 2_877_000_000,
        "neptune": 4_503_000_000,
    }

    d = distancia_sol_km.get(destino.lower())
    o = distancia_sol_km.get(origem.lower())

    if d is None or o is None:
        return {"erro": "Planeta inválido. Use nomes como Earth, Mars, Jupiter, etc."}

    # Distância aproximada entre órbitas (modelo simplificado)
    distancia_km = abs(d - o)

    # Velocidade da luz em km/s
    c_km_s = 299_792.458

    # Tempo em segundos e minutos
    tempo_s = distancia_km / c_km_s
    tempo_min = tempo_s / 60

    return {
        "tempo": round(tempo_s, 2),
        "distancia": distancia_km,
    }


In [53]:
# Criação de uma instância do agente utilizando o cliente Groq e a mensagem de sistema definida anteriormente.
agent = Agent(client, system_prompt)

In [54]:
# Exemplo de uso do agente para responder a uma pergunta sobre a massa da Terra multiplicada por 5, utilizando as ações definidas para obter a massa do planeta e realizar o cálculo necessário.
resultado = agent("Quanto tempo a luz leva para viajar de Mercúrio até a Terra?")
print(resultado)

Pensamento: Para calcular o tempo que a luz leva para viajar de Mercúrio até a Terra, preciso usar a ação disponível que calcula o tempo de viagem da luz entre dois corpos celestes. A ação "calcular_tempo_luz" parece ser a mais adequada para isso, considerando que Mercúrio é o planeta de origem e a Terra é o destino.

Ação: calcular_tempo_luz: Mercury, Earth
PAUSA


In [55]:
# Imprime as mensagens trocadas entre o agente e o modelo de linguagem, mostrando o processo de pensamento, ações realizadas e observações feitas durante a interação.
agent.messages

[{'role': 'system',
  'content': 'Você executa em um loop de Pensamento, Ação, PAUSA, Observação.\nAo final do loop você produz uma Resposta\nUse Pensamento para descrever seus pensamentos sobre a pergunta que você recebeu.\nUse Ação para executar uma das ações disponíveis para você - depois retorne PAUSA.\nObservação será o resultado da execução dessas ações.\n\nSuas ações disponíveis são:\n\ncalcular_funcao_2ograu:\nex. calcular_funcao_2ograu: a , b, c\nExecuta a fórmula de Bhaskara para resolver uma equação quadrática com os coeficientes a, b e c fornecidos e retorna o resultado\n\ncalcular_tempo_luz:\nex. calcular_tempo_luz: Mars, Earth\nCalcula o tempo que a luz leva para viajar entre a origem (padrão: Earth) e o planeta informado, retornando tempo em segundos e minutos e a distância aproximada.\n\nSessão de exemplo:\n\nPergunta: Quanto tempo a luz leva para viajar de Marte até a Terra?\nPensamento: Preciso calcular o tempo que a luz leva para viajar entre Marte e a Terra usando a

In [56]:
# Exemplo de uso da função get_planet_mass para obter a massa da Terra e imprimir o resultadoado.
obervation = calcular_tempo_luz("Earth","Mercury")
print(obervation)

{'tempo': 305.88, 'distancia': 91700000}


In [57]:
# Exemplo de uso do obersation para criar um novo prompt para o agente, mostrando a observação obtida e preparando o próximo passo da interação.
next_prompt = f"Observation: {obervation}"
print(next_prompt)

Observation: {'tempo': 305.88, 'distancia': 91700000}


In [58]:
# Exemplo de uso do agente para processar a nova mensagem contendo a observação obtida, permitindo que o agente continue o processo de pensamento e ação com base na nova informação.
next_prompt = f"Observação: {obervation}"
resultado = agent(next_prompt)
print(resultado)

Pensamento: Com a observação do resultado da ação "calcular_tempo_luz", obtenho os valores de tempo e distância. O tempo é de aproximadamente 305.88 segundos e a distância é de 91.700.000 km (ou 9.17 × 10^7 km). 

Agora, preciso converter o tempo em minutos para fornecer uma resposta mais completa e fácil de entender.

Resposta: A luz leva aproximadamente 305.88 segundos (ou cerca de 5.098 minutos) para viajar de Mercúrio até a Terra, cobrindo uma distância de aproximadamente 9.17 × 10^7 km.


In [59]:

# Imprime as mensagens trocadas entre o agente e o modelo de linguagem, mostrando o processo de pensamento, ações realizadas e observações feitas durante a interação.
agent.messages

[{'role': 'system',
  'content': 'Você executa em um loop de Pensamento, Ação, PAUSA, Observação.\nAo final do loop você produz uma Resposta\nUse Pensamento para descrever seus pensamentos sobre a pergunta que você recebeu.\nUse Ação para executar uma das ações disponíveis para você - depois retorne PAUSA.\nObservação será o resultado da execução dessas ações.\n\nSuas ações disponíveis são:\n\ncalcular_funcao_2ograu:\nex. calcular_funcao_2ograu: a , b, c\nExecuta a fórmula de Bhaskara para resolver uma equação quadrática com os coeficientes a, b e c fornecidos e retorna o resultado\n\ncalcular_tempo_luz:\nex. calcular_tempo_luz: Mars, Earth\nCalcula o tempo que a luz leva para viajar entre a origem (padrão: Earth) e o planeta informado, retornando tempo em segundos e minutos e a distância aproximada.\n\nSessão de exemplo:\n\nPergunta: Quanto tempo a luz leva para viajar de Marte até a Terra?\nPensamento: Preciso calcular o tempo que a luz leva para viajar entre Marte e a Terra usando a

In [60]:
# Exemplo de uso do agente para processar a nova mensagem contendo a observação obtida, permitindo que o agente continue o processo de pensamento e ação com base na nova informação.
next_prompt = f"Observação: {obervation}"
resultado = agent(next_prompt)
print(resultado)

Pensamento: Com a observação do resultado da ação "calcular_tempo_luz", obtenho os valores de tempo e distância. O tempo é de aproximadamente 305.88 segundos e a distância é de 91.700.000 km (ou 9.17 × 10^7 km). 

Agora, para fornecer uma resposta mais completa, posso converter o tempo para minutos. 305.88 segundos é igual a 305.88 / 60 = 5.098 minutos.

Resposta: A luz leva aproximadamente 305.88 segundos (ou cerca de 5.1 minutos) para viajar de Mercúrio até a Terra, cobrindo uma distância aproximada de 91.700.000 km.


In [61]:
# Imprime as mensagens trocadas entre o agente e o modelo de linguagem, mostrando o processo de pensamento, ações realizadas e observações feitas durante a interação.
agent.messages

[{'role': 'system',
  'content': 'Você executa em um loop de Pensamento, Ação, PAUSA, Observação.\nAo final do loop você produz uma Resposta\nUse Pensamento para descrever seus pensamentos sobre a pergunta que você recebeu.\nUse Ação para executar uma das ações disponíveis para você - depois retorne PAUSA.\nObservação será o resultado da execução dessas ações.\n\nSuas ações disponíveis são:\n\ncalcular_funcao_2ograu:\nex. calcular_funcao_2ograu: a , b, c\nExecuta a fórmula de Bhaskara para resolver uma equação quadrática com os coeficientes a, b e c fornecidos e retorna o resultado\n\ncalcular_tempo_luz:\nex. calcular_tempo_luz: Mars, Earth\nCalcula o tempo que a luz leva para viajar entre a origem (padrão: Earth) e o planeta informado, retornando tempo em segundos e minutos e a distância aproximada.\n\nSessão de exemplo:\n\nPergunta: Quanto tempo a luz leva para viajar de Marte até a Terra?\nPensamento: Preciso calcular o tempo que a luz leva para viajar entre Marte e a Terra usando a

In [62]:
import re

# Função para implementar o loop do agente, onde o agente processa uma mensagem de entrada, executa ações com base nas mensagens geradas e continua o processo até atingir um número máximo de iterações ou obter uma resposta final.
def agent_loop(iteracoes_max, system, query):
    # Cria uma instância nova do agente com o prompt de sistema
    agent = Agent(client, system)

    # Mapa de ferramentas permitidas (nome -> função)
    tools = {
        "calcular_tempo_luz": calcular_tempo_luz,
        "calcular_funcao_2ograu": calcular_funcao_2ograu,
    }

    # Primeiro prompt enviado ao agente
    next_prompt = query
    # Contador para evitar loop infinito
    i = 0

    # Executa até atingir o limite de iterações
    while i < iteracoes_max:
        i += 1

        # Envia o prompt atual e recebe resposta do modelo
        resultado = agent(next_prompt)
        print(resultado)

        # Só tenta executar tool se o modelo indicou PAUSA + Ação
        # Regex aceita "Ação" e "Acao", com variação de caixa
        if "PAUSA" in resultado.upper() and re.search(r"A[cç][aã]o:", resultado, re.IGNORECASE):
            # Extrai: nome da ferramenta + argumentos
            # Exemplo esperado: "Ação: calcular_tempo_luz: Mars, Earth"
            match = re.search(r"A[cç][aã]o:\s*([a-z0-9_]+)\s*:\s*(.*)", resultado, re.IGNORECASE)

            # Se o formato não bater, devolve observação para o agente corrigir
            if not match:
                next_prompt = "Observação: Não consegui interpretar a ação. Use formato: Ação: nome_tool: argumentos"
                print(next_prompt)
                continue

            # Nome da tool e string de argumentos extraídos da resposta
            chosen_tool = match.group(1).strip()
            arg = match.group(2).strip()

            # Busca a função correspondente no dicionário
            tool_fn = tools.get(chosen_tool)
            if not tool_fn:
                # Tool inexistente: avisa o agente via observação
                next_prompt = f"Observação: Ferramenta '{chosen_tool}' não encontrada"
                print(next_prompt)
                continue

            try:
                # Executa a tool mantendo contrato atual (1 string de entrada)
                resultado_tool = tool_fn(arg)
                # Retorna resultado da tool como observação para próxima iteração
                next_prompt = f"Observação: {resultado_tool}"
            except Exception as e:
                # Captura falhas de execução e devolve erro ao agente
                next_prompt = f"Observação: erro ao executar '{chosen_tool}': {e}"

            print(next_prompt)
            continue

        # Encerra o loop quando o modelo já trouxe resposta final
        if "Resposta:" in resultado:
            break

In [64]:
# Chama a função de loop do agente com um número máximo de iterações, o prompt de sistema definido e uma pergunta inicial sobre o tempo para luz chegar em Netuno.
agent_loop(iteracoes_max=10, system=system_prompt, query="Quanto tempo a luz leva para viajar de Vênus até a Netuno?")

Pensamento: Para calcular o tempo que a luz leva para viajar de Vênus até Netuno, preciso considerar a ação disponível que calcula o tempo que a luz leva para viajar entre dois planetas. No entanto, a ação "calcular_tempo_luz" parece calcular o tempo em relação à Terra. Portanto, preciso adaptar a abordagem para considerar a distância entre Vênus e Netuno, mas como a ação disponível se baseia em relação à Terra, vou proceder com a esperança de que a ação seja flexível o suficiente para lidar com isso de alguma forma, mesmo que não seja exatamente como descrito.

Ação: calcular_tempo_luz: Venus, Neptune
PAUSA
Observação: {'tempo': 14659.47, 'distancia': 4394800000}
Pensamento: Com base na observação, obtenho o tempo e a distância que a luz leva para viajar de Vênus até Netuno. O tempo é de aproximadamente 14659.47 segundos e a distância é de 4,3948 bilhões de quilômetros. Preciso convertar o tempo em minutos para tornar a resposta mais compreensível.

Ação: calcular_funcao_2ograu: 1, 0,

In [ ]:
# Chama a função de loop do agente com um número máximo de iterações, o prompt de sistema definido e uma pergunta inicial sobre a soma de raizes de uma equação do 2º grau.
agent_loop(iteracoes_max=10, system=system_prompt, query="Quero que você some as raizes da equação x^2 + 3x – 4 = 0")

Pensamento: Para somar as raízes da equação x^2 + 3x – 4 = 0, preciso primeiro encontrar as raízes. Posso usar a fórmula de Bhaskara para resolver a equação quadrática. A fórmula de Bhaskara é dada por x = (-b ± √(b^2 - 4ac)) / 2a, onde a, b e c são os coeficientes da equação. Nesse caso, a = 1, b = 3 e c = -4.

Ação: calcular_funcao_2ograu: 1, 3, -4
PAUSA
Observação: {'delta': 25.0, 'x1': 1.0, 'x2': -4.0}
Pensamento: Agora que tenho as raízes da equação, x1 = 1 e x2 = -4, posso somá-las para encontrar a resposta. A soma das raízes é simplesmente x1 + x2.

Ação: Não preciso executar nenhuma ação adicional, apenas calcular a soma das raízes.

Resposta: A soma das raízes da equação x^2 + 3x – 4 = 0 é 1 + (-4) = -3.
